In [1]:
%pip install -U sentence-transformers

print("Sentence Transformers installation completed.")

  Using cached sentence_transformers-6.0.0-py3-none-any.whl.metadata (20 kB)
  Using cached transformers-5.16.1-py3-none-any.whl.metadata (32 kB)
  Using cached tokenizers-0.23.1-cp310-abi3-win_amd64.whl.metadata (10 kB)
  Using cached huggingface_hub-1.29.0-py3-none-any.whl.metadata (16 kB)
  Using cached torch-2.13.0-cp311-cp311-win_amd64.whl.metadata (39 kB)
  Using cached scikit_learn-1.9.0-cp311-cp311-win_amd64.whl.metadata (11 kB)
  Using cached scipy-1.17.1-cp311-cp311-win_amd64.whl.metadata (60 kB)
  Using cached click-8.5.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached filelock-3.32.4-py3-none-any.whl.metadata (2.0 kB)
  Using cached fsspec-2026.7.0-py3-none-any.whl.metadata (10 kB)
  Using cached hf_xet-1.6.0-cp38-abi3-win_amd64.whl.metadata (4.9 kB)
  Using cached pyyaml-6.0.3-cp311-cp311-win_amd64.whl.metadata (2.4 kB)
  Using cached regex-2026.7.19-cp311-cp311-win_amd64.whl.metadata (41 kB)
  Using cached typer-0.27.2-py3-none-any.whl.metadata (16 kB)
  Using cached sa

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [1]:
# ENVIRONMENT & CONFIGURATION

import os
import json
import time
import numpy as np

from pathlib import Path

print("=" * 60)
print("SWIFTBASKET — LOCAL EMBEDDING PIPELINE")
print("=" * 60)

print("Python environment loaded successfully.")
print("Local embedding pipeline initialized.")

SWIFTBASKET — LOCAL EMBEDDING PIPELINE
Python environment loaded successfully.
Local embedding pipeline initialized.


In [11]:
# LOCAL EMBEDDING MODEL CONFIGURATION

LOCAL_MODEL_NAME = "BAAI/bge-small-en-v1.5"

# BGE-small-en-v1.5 produces 384-dimensional embeddings.
EMBEDDING_DIMENSION = 384

# Conservative starting batch size.
BATCH_SIZE = 32

CHECKPOINT_INTERVAL = 1

print("=" * 60)
print("LOCAL EMBEDDING MODEL CONFIGURATION")
print("=" * 60)

print(f"Model               : {LOCAL_MODEL_NAME}")
print(f"Embedding dimension : {EMBEDDING_DIMENSION}")
print(f"Batch size          : {BATCH_SIZE}")
print("Device              : CPU")
print("API dependency      : NONE")

LOCAL EMBEDDING MODEL CONFIGURATION
Model               : BAAI/bge-small-en-v1.5
Embedding dimension : 384
Batch size          : 32
Device              : CPU
API dependency      : NONE


In [4]:
# DATA & OUTPUT PATHS

BASE_DIR = Path(r"E:\Python Projects\SwiftBaSkeT-AI\data")

# Source corpora
CORPUS_FILES = {
    "products": BASE_DIR / "products.jsonl",
    "orders": BASE_DIR / "orders.jsonl",
    "returns": BASE_DIR / "returns.jsonl"
}

# NEW LOCAL EMBEDDING OUTPUTS
LOCAL_OUTPUT_FILE = BASE_DIR / "local_mvp_embeddings.jsonl"
LOCAL_CHECKPOINT_FILE = BASE_DIR / "local_embedding_progress.json"

# Old Gemini output is deliberately NOT used.
GEMINI_OUTPUT_FILE = BASE_DIR / "mvp_embeddings.jsonl"

print("=" * 60)
print("DATA & OUTPUT PATH CONFIGURATION")
print("=" * 60)

print(f"Base directory       : {BASE_DIR}")
print(f"Products             : {CORPUS_FILES['products']}")
print(f"Orders               : {CORPUS_FILES['orders']}")
print(f"Returns              : {CORPUS_FILES['returns']}")
print(f"Local output         : {LOCAL_OUTPUT_FILE}")
print(f"Local checkpoint     : {LOCAL_CHECKPOINT_FILE}")
print(f"Gemini output        : {GEMINI_OUTPUT_FILE}")

print("\nPath configuration completed.")

DATA & OUTPUT PATH CONFIGURATION
Base directory       : E:\Python Projects\SwiftBaSkeT-AI\data
Products             : E:\Python Projects\SwiftBaSkeT-AI\data\products.jsonl
Orders               : E:\Python Projects\SwiftBaSkeT-AI\data\orders.jsonl
Returns              : E:\Python Projects\SwiftBaSkeT-AI\data\returns.jsonl
Local output         : E:\Python Projects\SwiftBaSkeT-AI\data\local_mvp_embeddings.jsonl
Local checkpoint     : E:\Python Projects\SwiftBaSkeT-AI\data\local_embedding_progress.json
Gemini output        : E:\Python Projects\SwiftBaSkeT-AI\data\mvp_embeddings.jsonl

Path configuration completed.


In [5]:
# LOAD SOURCE CORPUS

EXPECTED_COUNTS = {
    "products": 2314,
    "orders": 10000,
    "returns": 28866
}

corpora = {}

for corpus_name, filepath in CORPUS_FILES.items():

    print("\n" + "-" * 60)
    print(f"LOADING {corpus_name.upper()}...")
    print(f"Path: {filepath}")

    if not filepath.exists():
        raise FileNotFoundError(
            f"File not found: {filepath}"
        )

    documents = []

    with open(
        filepath,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            line = line.strip()

            if not line:
                continue

            documents.append(
                json.loads(line)
            )

    corpora[corpus_name] = documents

    print(
        f"Loaded documents: "
        f"{len(documents):,}"
    )

    expected = EXPECTED_COUNTS[corpus_name]

    if len(documents) != expected:
        raise ValueError(
            f"{corpus_name} count mismatch. "
            f"Expected {expected:,}, "
            f"got {len(documents):,}"
        )

    print("Count verification: PASS")


print("\n" + "=" * 60)
print("SOURCE CORPUS LOADING COMPLETED")
print("=" * 60)

for corpus_name, documents in corpora.items():
    print(
        f"{corpus_name.capitalize():10} : "
        f"{len(documents):,}"
    )

total_source_documents = sum(
    len(documents)
    for documents in corpora.values()
)

print(
    f"{'TOTAL':10} : "
    f"{total_source_documents:,}"
)


------------------------------------------------------------
LOADING PRODUCTS...
Path: E:\Python Projects\SwiftBaSkeT-AI\data\products.jsonl
Loaded documents: 2,314
Count verification: PASS

------------------------------------------------------------
LOADING ORDERS...
Path: E:\Python Projects\SwiftBaSkeT-AI\data\orders.jsonl
Loaded documents: 10,000
Count verification: PASS

------------------------------------------------------------
LOADING RETURNS...
Path: E:\Python Projects\SwiftBaSkeT-AI\data\returns.jsonl
Loaded documents: 28,866
Count verification: PASS

SOURCE CORPUS LOADING COMPLETED
Products   : 2,314
Orders     : 10,000
Returns    : 28,866
TOTAL      : 41,180


In [6]:
# MVP CORPUS SELECTION

mvp_documents = (
    corpora["products"]
    + corpora["orders"]
)

all_documents = mvp_documents

print("=" * 60)
print("SWIFTBASKET — MVP CORPUS")
print("=" * 60)

print(
    f"Products included : "
    f"{len(corpora['products']):,}"
)

print(
    f"Orders included   : "
    f"{len(corpora['orders']):,}"
)

print(
    f"Returns included  : "
    f"{len(corpora['returns']):,}"
    " → NOT IN MVP EMBEDDING"
)

print("-" * 60)

print(
    f"Total MVP documents : "
    f"{len(all_documents):,}"
)

EXPECTED_MVP_COUNT = 12314

if len(all_documents) != EXPECTED_MVP_COUNT:
    raise ValueError(
        f"MVP count mismatch. "
        f"Expected {EXPECTED_MVP_COUNT:,}, "
        f"got {len(all_documents):,}"
    )

print("\nMVP count verification: PASS")

SWIFTBASKET — MVP CORPUS
Products included : 2,314
Orders included   : 10,000
Returns included  : 28,866 → NOT IN MVP EMBEDDING
------------------------------------------------------------
Total MVP documents : 12,314

MVP count verification: PASS


In [7]:
# MVP DOCUMENT VALIDATION

print("=" * 60)
print("MVP DOCUMENT VALIDATION")
print("=" * 60)

required_fields = [
    "id",
    "text",
    "metadata"
]

invalid_documents = []

for index, doc in enumerate(all_documents):

    missing_fields = [
        field
        for field in required_fields
        if field not in doc
    ]

    if missing_fields:
        invalid_documents.append(
            {
                "index": index,
                "id": doc.get("id"),
                "missing": missing_fields
            }
        )

if invalid_documents:

    print(
        f"Invalid documents: "
        f"{len(invalid_documents):,}"
    )

    print(
        "First invalid document:"
    )

    print(invalid_documents[0])

    raise ValueError(
        "MVP document validation failed."
    )

print(
    f"Documents validated : "
    f"{len(all_documents):,}"
)

print("Required fields      : PASS")
print("Document validation  : PASS")

MVP DOCUMENT VALIDATION
Documents validated : 12,314
Required fields      : PASS
Document validation  : PASS


In [8]:
# MVP CORPUS CHARACTERISTICS

print("=" * 60)
print("MVP CORPUS CHARACTERISTICS")
print("=" * 60)

text_lengths = [
    len(str(doc.get("text", "")))
    for doc in all_documents
]

print(
    f"Documents       : "
    f"{len(all_documents):,}"
)

print(
    f"Min text chars  : "
    f"{min(text_lengths):,}"
)

print(
    f"Max text chars  : "
    f"{max(text_lengths):,}"
)

print(
    f"Average chars   : "
    f"{sum(text_lengths) / len(text_lengths):,.0f}"
)

print(
    f"Total characters: "
    f"{sum(text_lengths):,}"
)

print("\nCorpus characteristics verification: PASS")

MVP CORPUS CHARACTERISTICS
Documents       : 12,314
Min text chars  : 412
Max text chars  : 3,834
Average chars   : 933
Total characters: 11,485,938

Corpus characteristics verification: PASS


In [9]:
# LOAD LOCAL EMBEDDING MODEL

from sentence_transformers import SentenceTransformer

print("=" * 60)
print("LOADING LOCAL EMBEDDING MODEL")
print("=" * 60)

print(f"Model : {LOCAL_MODEL_NAME}")
print("Device: CPU")
print("\nLoading model...")

local_model = SentenceTransformer(
    LOCAL_MODEL_NAME,
    device="cpu"
)

print("\nLocal embedding model loaded successfully.")

E:\conda_envs\swiftbasket-ai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LOADING LOCAL EMBEDDING MODEL
Model : BAAI/bge-small-en-v1.5
Device: CPU

Loading model...


E:\conda_envs\swiftbasket-ai\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\VICKY\.cache\huggingface\hub\models--BAAI--bge-small-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 527.89it/s]



Local embedding model loaded successfully.


In [12]:
# SINGLE DOCUMENT LOCAL EMBEDDING TEST

test_document = all_documents[0]["text"]

test_embedding = local_model.encode(
    test_document,
    normalize_embeddings=True,
    show_progress_bar=False
)

print("=" * 60)
print("LOCAL EMBEDDING TEST")
print("=" * 60)

print(
    f"Model               : "
    f"{LOCAL_MODEL_NAME}"
)

print(
    f"Expected dimensions : "
    f"{EMBEDDING_DIMENSION}"
)

print(
    f"Actual dimensions   : "
    f"{len(test_embedding)}"
)

print(
    f"First 5 values      : "
    f"{test_embedding[:5]}"
)

if len(test_embedding) != EMBEDDING_DIMENSION:

    raise ValueError(
        f"Dimension mismatch. "
        f"Expected {EMBEDDING_DIMENSION}, "
        f"got {len(test_embedding)}"
    )

print("\nDimension verification: PASS")

LOCAL EMBEDDING TEST
Model               : BAAI/bge-small-en-v1.5
Expected dimensions : 384
Actual dimensions   : 384
First 5 values      : [-0.05973984 -0.01535011  0.02162428 -0.0369458   0.02928796]

Dimension verification: PASS


In [13]:
# LOCAL BATCH EMBEDDING TEST

test_documents = [
    doc["text"]
    for doc in all_documents[:5]
]

test_embeddings = local_model.encode(
    test_documents,
    batch_size=5,
    normalize_embeddings=True,
    show_progress_bar=False
)

print("=" * 60)
print("LOCAL BATCH EMBEDDING TEST")
print("=" * 60)

print(
    f"Batch size          : "
    f"{len(test_documents)}"
)

print(
    f"Embedding dimension : "
    f"{test_embeddings.shape[1]}"
)

print(
    f"Embeddings returned : "
    f"{len(test_embeddings)}"
)

for i, (doc, embedding) in enumerate(
    zip(
        all_documents[:5],
        test_embeddings
    ),
    start=1
):

    print(
        f"Document {i}: "
        f"{doc['id']} | "
        f"Dimensions: {len(embedding)} | "
        f"First 3: {embedding[:3].tolist()}"
    )

if len(test_embeddings) != 5:
    raise ValueError(
        "Incorrect number of embeddings returned."
    )

if test_embeddings.shape[1] != EMBEDDING_DIMENSION:
    raise ValueError(
        "Embedding dimension mismatch."
    )

print("\nLOCAL BATCH EMBEDDING TEST: PASS")

LOCAL BATCH EMBEDDING TEST
Batch size          : 5
Embedding dimension : 384
Embeddings returned : 5
Document 1: P000001 | Dimensions: 384 | First 3: [-0.05973980948328972, -0.015350095927715302, 0.021624239161610603]
Document 2: P000002 | Dimensions: 384 | First 3: [-0.0456218495965004, -0.014106938615441322, 0.007066698744893074]
Document 3: P000003 | Dimensions: 384 | First 3: [-0.059917256236076355, -0.010574770160019398, 0.018302129581570625]
Document 4: P000004 | Dimensions: 384 | First 3: [-0.09396453201770782, 0.020808754488825798, 0.01928265579044819]
Document 5: P000005 | Dimensions: 384 | First 3: [-0.02876700647175312, -0.023142458871006966, 0.0025152855087071657]

LOCAL BATCH EMBEDDING TEST: PASS


In [14]:
# LOCAL EMBEDDING BENCHMARK

BENCHMARK_SIZE = 100

benchmark_documents = [
    doc["text"]
    for doc in all_documents[:BENCHMARK_SIZE]
]

print("=" * 60)
print("LOCAL EMBEDDING BENCHMARK")
print("=" * 60)

print(
    f"Documents to process : "
    f"{BENCHMARK_SIZE}"
)

print(
    f"Batch size           : "
    f"{BATCH_SIZE}"
)

print("\nBenchmark running...\n")

start_time = time.time()

benchmark_embeddings = local_model.encode(
    benchmark_documents,
    batch_size=BATCH_SIZE,
    normalize_embeddings=True,
    show_progress_bar=True
)

elapsed_seconds = time.time() - start_time

documents_per_second = (
    BENCHMARK_SIZE / elapsed_seconds
)

estimated_seconds = (
    12314 / documents_per_second
)

estimated_minutes = (
    estimated_seconds / 60
)

print("\n" + "=" * 60)
print("BENCHMARK RESULTS")
print("=" * 60)

print(
    f"Documents processed : "
    f"{BENCHMARK_SIZE}"
)

print(
    f"Time taken          : "
    f"{elapsed_seconds:.2f} seconds"
)

print(
    f"Documents / second  : "
    f"{documents_per_second:.2f}"
)

print(
    f"Estimated time for 12,314 : "
    f"{estimated_minutes:.2f} minutes"
)

print("\nBenchmark completed.")

LOCAL EMBEDDING BENCHMARK
Documents to process : 100
Batch size           : 32

Benchmark running...



Batches: 100%|██████████| 4/4 [00:05<00:00,  1.39s/it]


BENCHMARK RESULTS
Documents processed : 100
Time taken          : 5.67 seconds
Documents / second  : 17.63
Estimated time for 12,314 : 11.64 minutes

Benchmark completed.


In [15]:
# CHECKPOINT & RESUME MANAGEMENT

def load_local_checkpoint():

    if LOCAL_CHECKPOINT_FILE.exists():

        with open(
            LOCAL_CHECKPOINT_FILE,
            "r",
            encoding="utf-8"
        ) as f:

            checkpoint = json.load(f)

        print("Existing local checkpoint loaded.")

        return checkpoint

    checkpoint = {
        "status": "NOT_STARTED",
        "completed_documents": 0,
        "completed_batches": 0,
        "model": LOCAL_MODEL_NAME,
        "embedding_dimension": EMBEDDING_DIMENSION
    }

    print("No existing local checkpoint found.")

    return checkpoint


def save_local_checkpoint(checkpoint):

    with open(
        LOCAL_CHECKPOINT_FILE,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            checkpoint,
            f,
            indent=4
        )


def get_existing_local_embedding_ids():

    existing_ids = set()

    if not LOCAL_OUTPUT_FILE.exists():

        return existing_ids

    with open(
        LOCAL_OUTPUT_FILE,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            line = line.strip()

            if not line:
                continue

            record = json.loads(line)

            existing_ids.add(
                str(record["id"])
            )

    return existing_ids


checkpoint = load_local_checkpoint()

existing_local_ids = (
    get_existing_local_embedding_ids()
)

print("\n" + "=" * 60)
print("LOCAL CHECKPOINT STATUS")
print("=" * 60)

print(
    f"Existing embeddings : "
    f"{len(existing_local_ids):,}"
)

print(
    f"Total MVP documents : "
    f"{len(all_documents):,}"
)

print(
    f"Remaining documents : "
    f"{len(all_documents) - len(existing_local_ids):,}"
)

No existing local checkpoint found.

LOCAL CHECKPOINT STATUS
Existing embeddings : 0
Total MVP documents : 12,314
Remaining documents : 12,314


In [16]:
# LOCAL EMBEDDING ENGINE

def generate_local_embeddings(
    texts,
    batch_size=BATCH_SIZE
):

    embeddings = local_model.encode(
        texts,
        batch_size=batch_size,
        normalize_embeddings=True,
        show_progress_bar=False,
        convert_to_numpy=True
    )

    return embeddings.tolist()


print("Local embedding engine initialized successfully.")
print("No API retry logic required.")

Local embedding engine initialized successfully.
No API retry logic required.


In [17]:
# MVP LOCAL PRODUCTION EMBEDDING RUN

print("=" * 60)
print("SWIFTBASKET — LOCAL MVP EMBEDDING RUN")
print("=" * 60)

checkpoint = load_local_checkpoint()

existing_ids = (
    get_existing_local_embedding_ids()
)

print(
    f"Total MVP documents : "
    f"{len(all_documents):,}"
)

print(
    f"Embedding dimension : "
    f"{EMBEDDING_DIMENSION}"
)

print(
    f"Batch size          : "
    f"{BATCH_SIZE}"
)

print(
    f"Model               : "
    f"{LOCAL_MODEL_NAME}"
)

print(
    f"\nExisting embeddings : "
    f"{len(existing_ids):,}"
)

remaining_documents = [
    doc
    for doc in all_documents
    if str(doc["id"]) not in existing_ids
]

print(
    f"Remaining documents : "
    f"{len(remaining_documents):,}"
)

if not remaining_documents:

    print("\nAll MVP documents are already embedded.")

    checkpoint["status"] = "COMPLETED"
    checkpoint["completed_documents"] = (
        len(existing_ids)
    )

    save_local_checkpoint(checkpoint)

else:

    batches = [
        remaining_documents[i:i + BATCH_SIZE]
        for i in range(
            0,
            len(remaining_documents),
            BATCH_SIZE
        )
    ]

    print(
        f"Remaining batches   : "
        f"{len(batches):,}"
    )

    checkpoint["status"] = "RUNNING"

    save_local_checkpoint(checkpoint)

    for batch_number, batch in enumerate(
        batches,
        start=1
    ):

        print("\n" + "-" * 60)

        print(
            f"LOCAL BATCH "
            f"{batch_number}/{len(batches)}"
        )

        print(
            f"Documents : "
            f"{len(batch)}"
        )

        texts = [
            doc["text"]
            for doc in batch
        ]

        try:

            start_time = time.time()

            embeddings = generate_local_embeddings(
                texts,
                batch_size=BATCH_SIZE
            )

            elapsed = (
                time.time() - start_time
            )

            if len(embeddings) != len(batch):

                raise ValueError(
                    "Embedding count does not "
                    "match document count."
                )

            for embedding in embeddings:

                if len(embedding) != EMBEDDING_DIMENSION:

                    raise ValueError(
                        "Embedding dimension mismatch."
                    )

            with open(
                LOCAL_OUTPUT_FILE,
                "a",
                encoding="utf-8"
            ) as f:

                for doc, embedding in zip(
                    batch,
                    embeddings
                ):

                    record = {
                        "id": str(doc["id"]),
                        "text": doc["text"],
                        "embedding": embedding,
                        "metadata": doc["metadata"]
                    }

                    f.write(
                        json.dumps(
                            record,
                            ensure_ascii=False
                        )
                        + "\n"
                    )

            # Update completed IDs
            for doc in batch:

                existing_ids.add(
                    str(doc["id"])
                )

            checkpoint[
                "completed_documents"
            ] = len(existing_ids)

            checkpoint[
                "completed_batches"
            ] = (
                checkpoint.get(
                    "completed_batches",
                    0
                ) + 1
            )

            checkpoint["status"] = "RUNNING"

            save_local_checkpoint(
                checkpoint
            )

            print(
                f"SUCCESS: "
                f"{len(batch)} embeddings written."
            )

            print(
                f"Time taken : "
                f"{elapsed:.2f} seconds"
            )

            print(
                f"Total completed: "
                f"{len(existing_ids):,}/"
                f"{len(all_documents):,}"
            )

        except Exception as e:

            checkpoint["status"] = "FAILED"

            checkpoint[
                "completed_documents"
            ] = len(existing_ids)

            save_local_checkpoint(
                checkpoint
            )

            print("\n" + "=" * 60)
            print("LOCAL EMBEDDING RUN STOPPED")
            print("=" * 60)

            print(
                f"Completed embeddings : "
                f"{len(existing_ids):,}"
            )

            print(
                f"Remaining            : "
                f"{len(all_documents) - len(existing_ids):,}"
            )

            print(
                f"\nError: {e}"
            )

            print(
                "\nCheckpoint saved."
            )

            print(
                "You can safely rerun this cell."
            )

            raise

    checkpoint["status"] = "COMPLETED"

    checkpoint[
        "completed_documents"
    ] = len(existing_ids)

    save_local_checkpoint(
        checkpoint
    )

    print("\n" + "=" * 60)
    print("LOCAL MVP EMBEDDING RUN COMPLETED")
    print("=" * 60)

    print(
        f"Total embeddings : "
        f"{len(existing_ids):,}"
    )

SWIFTBASKET — LOCAL MVP EMBEDDING RUN
No existing local checkpoint found.
Total MVP documents : 12,314
Embedding dimension : 384
Batch size          : 32
Model               : BAAI/bge-small-en-v1.5

Existing embeddings : 0
Remaining documents : 12,314
Remaining batches   : 385

------------------------------------------------------------
LOCAL BATCH 1/385
Documents : 32
SUCCESS: 32 embeddings written.
Time taken : 1.72 seconds
Total completed: 32/12,314

------------------------------------------------------------
LOCAL BATCH 2/385
Documents : 32
SUCCESS: 32 embeddings written.
Time taken : 1.80 seconds
Total completed: 64/12,314

------------------------------------------------------------
LOCAL BATCH 3/385
Documents : 32
SUCCESS: 32 embeddings written.
Time taken : 1.61 seconds
Total completed: 96/12,314

------------------------------------------------------------
LOCAL BATCH 4/385
Documents : 32
SUCCESS: 32 embeddings written.
Time taken : 2.01 seconds
Total completed: 128/12,314


In [18]:
# FINAL LOCAL EMBEDDING VALIDATION

print("=" * 60)
print("FINAL LOCAL EMBEDDING VALIDATION")
print("=" * 60)

if not LOCAL_OUTPUT_FILE.exists():

    raise FileNotFoundError(
        "Local embedding output file does not exist."
    )

embedding_count = 0
invalid_dimensions = 0

first_record = None
last_record = None

with open(
    LOCAL_OUTPUT_FILE,
    "r",
    encoding="utf-8"
) as f:

    for line in f:

        line = line.strip()

        if not line:
            continue

        record = json.loads(line)

        embedding_count += 1

        if first_record is None:
            first_record = record

        last_record = record

        if len(record["embedding"]) != EMBEDDING_DIMENSION:

            invalid_dimensions += 1


print(
    f"Expected embeddings : "
    f"{len(all_documents):,}"
)

print(
    f"Actual embeddings   : "
    f"{embedding_count:,}"
)

print(
    f"Invalid dimensions  : "
    f"{invalid_dimensions:,}"
)

if embedding_count != len(all_documents):

    raise ValueError(
        "Final embedding count does not "
        "match MVP document count."
    )

if invalid_dimensions != 0:

    raise ValueError(
        "One or more embeddings have "
        "incorrect dimensions."
    )

print("\n" + "-" * 60)

print("FINAL VERIFICATION: PASS")

print(
    f"Documents embedded : "
    f"{embedding_count:,}"
)

print(
    f"Vector dimensions   : "
    f"{EMBEDDING_DIMENSION}"
)

print(
    f"Output file         : "
    f"{LOCAL_OUTPUT_FILE}"
)

print("=" * 60)

FINAL LOCAL EMBEDDING VALIDATION
Expected embeddings : 12,314
Actual embeddings   : 12,314
Invalid dimensions  : 0

------------------------------------------------------------
FINAL VERIFICATION: PASS
Documents embedded : 12,314
Vector dimensions   : 384
Output file         : E:\Python Projects\SwiftBaSkeT-AI\data\local_mvp_embeddings.jsonl
